**Data Cleaner**



In [1]:
import pandas as pd

class dataCleaner:
    def __init__(self, csv_file):
        # ENSURE YOU UPLOAD CSV FILE FIRST, USING FOLDER ICON
        # You will need to start the session first before adding the csv.
        self.df = pd.read_csv(csv_file)

        self.copy = self.df.copy()
        pass

    def clean_data(self):
        # Removes rows containing Preseason games
        self.copy = self.copy[~self.copy['Week'].str.contains('Preseason', na=False)]

        # Removing Hall of Fame games
        self.copy = self.copy[~self.copy['Week'].str.contains('Hall Of Fame', na=False)]

        # Removing BYE weeks
        self.copy = self.copy[~self.copy['GameStatus'].str.contains('BYE', na=False)]

        # Removing Canceled Games
        self.copy = self.copy[~self.copy['Week'].str.contains('CANCELLED', na=False)]

        # Removing Pro Bowl Games
        self.copy = self.copy[~self.copy['Week'].str.contains('Pro Bowl', na=False)]

        # Reset Index after Filtering
        self.copy.reset_index(drop=True, inplace=True)

        pass

    def parse_dates(self):
        # 1. Combine 'Date' and 'Season' (e.g. "01/05" + "2017")
        date_strs = self.copy['Date'].astype(str) + '/' + self.copy['Season'].astype(str)

        # 2. Convert to datetime
        self.copy['FullDate'] = pd.to_datetime(date_strs, format='%m/%d/%Y', errors='coerce')

        # 3. Fix Jan/Feb/Mar games (Add 1 year)
        mask = self.copy['FullDate'].dt.month <= 3
        self.copy.loc[mask, 'FullDate'] = self.copy.loc[mask, 'FullDate'] + pd.DateOffset(years=1)

        # 4. Sort by date
        self.copy = self.copy.sort_values('FullDate')

    def split_record(self, record_str):
        if pd.isna(record_str):
            return 0, 0

        parts = str(record_str).split('-')
        if len(parts) >= 2:
            return int(parts[0]), int(parts[1])
        return 0, 0

    def parse_records_and_scores(self):
        # Create the new columns
        temp_away = self.copy['AwayRecord'].apply(self.split_record)
        self.copy['AwayWins'], self.copy['AwayLosses'] = zip(*temp_away)

        temp_home = self.copy['HomeRecord'].apply(self.split_record)
        self.copy['HomeWins'], self.copy['HomeLosses'] = zip(*temp_home)

        # DROP the old columns with the dashes
        self.copy.drop(columns=['AwayRecord', 'HomeRecord'], inplace=True)

    def get_head(self):
        return self.copy.head()

    def save_file(self, new_filename):
        self.copy.to_csv(new_filename, index=False)
        print(f"Success! Saved the full file to: {new_filename}")

    def calculate_rest_days(self):
        # Ensure dates are datetime objects
        if 'FullDate' in self.copy.columns:
            # Use FullDate if parse_dates() was already run
            pass
        else:
             # Fallback if parse_dates wasn't run, though it should be
            self.copy['FullDate'] = pd.to_datetime(self.copy['Date'] + '/' + self.copy['Season'].astype(str), format='%m/%d/%Y', errors='coerce')

        # Sort by date to ensure chronological order
        self.copy = self.copy.sort_values('FullDate')

        # Dictionaries to store the date of the last game played for each team
        last_game_date = {}

        # Lists to store the calculated rest days
        home_rest = []
        away_rest = []

        # Standard rest for first game of season (e.g., 7 days)
        DEFAULT_REST = 7

        for index, row in self.copy.iterrows():
            current_date = row['FullDate']
            home_team = row['HomeTeam']
            away_team = row['AwayTeam']

            # --- Calculate Home Rest ---
            if home_team in last_game_date:
                delta = (current_date - last_game_date[home_team]).days
                home_rest.append(delta)
            else:
                home_rest.append(DEFAULT_REST)

            # --- Calculate Away Rest ---
            if away_team in last_game_date:
                delta = (current_date - last_game_date[away_team]).days
                away_rest.append(delta)
            else:
                away_rest.append(DEFAULT_REST)

            # --- Update Last Game Dates ---
            last_game_date[home_team] = current_date
            last_game_date[away_team] = current_date

        self.copy['HomeRestDays'] = home_rest
        self.copy['AwayRestDays'] = away_rest


dC = dataCleaner("2017-2025_scores.csv")
dC.clean_data()
dC.parse_records_and_scores()
dC.calculate_rest_days()

print(dC.get_head())
dC.save_file("2017-2025_scores_cleaned.csv")

     Season                 Week GameStatus  Day   Date  AwayTeam  AwayScore  \
256    2017            Wild Card      FINAL  SAT  01/06    Titans       22.0   
257    2017            Wild Card      FINAL  SAT  01/06   Falcons       26.0   
258    2017            Wild Card      FINAL  SUN  01/07     Bills        3.0   
259    2017            Wild Card      FINAL  SUN  01/07  Panthers       26.0   
260    2017  Divisional Playoffs      FINAL  SAT  01/13   Falcons       10.0   

     AwayWin HomeTeam  HomeScore  ...  AwaySeeding  HomeSeeding  PostSeason  \
256      1.0   Chiefs       21.0  ...          5.0          4.0           1   
257      1.0     Rams       13.0  ...          6.0          3.0           1   
258      0.0  Jaguars       10.0  ...          6.0          3.0           1   
259      0.0   Saints       31.0  ...          5.0          4.0           1   
260      0.0   Eagles       15.0  ...          6.0          1.0           1   

     AwayWins  AwayLosses  HomeWins  HomeLos

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


**Feature Engineering**

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler

class FeatureEngineer:
    def __init__(self, clean_df):
      csv =  pd.read_csv(clean_df)
      self.df = csv.copy()

      if 'FullDate' in self.df.columns:
            self.df['FullDate'] = pd.to_datetime(self.df['FullDate'])

    # Brandon
    def create_rolling_stats(self, window=5):
        """
        Calculates the average points scored and allowed over the last 'window' games.
        """
        print(f"Calculating {window}-game rolling stats...")

        # 1. Create a "Long" dataframe.
        # CHANGED: Keep 'FullDate' instead of renaming to 'Date' to avoid collision with original 'Date' column
        home_df = self.df[['FullDate', 'HomeTeam', 'HomeScore', 'AwayScore']].copy()
        home_df.columns = ['FullDate', 'Team', 'PointsScored', 'PointsAllowed']

        away_df = self.df[['FullDate', 'AwayTeam', 'AwayScore', 'HomeScore']].copy()
        away_df.columns = ['FullDate', 'Team', 'PointsScored', 'PointsAllowed']

        # Combine and Sort
        team_stats = pd.concat([home_df, away_df]).sort_values(['Team', 'FullDate'])

        # 2. Calculate Rolling Averages
        # .shift(1) is CRITICAL: We want the stats *entering* the game, not *including* the game
        team_stats['AvgPointsScored'] = team_stats.groupby('Team')['PointsScored'] \
                                        .transform(lambda x: x.rolling(window, min_periods=1).mean().shift(1))

        team_stats['AvgPointsAllowed'] = team_stats.groupby('Team')['PointsAllowed'] \
                                        .transform(lambda x: x.rolling(window, min_periods=1).mean().shift(1))

        # 3. Merge back to the main DataFrame

        # Merge Home Stats
        # CHANGED: right_on uses 'FullDate'
        self.df = self.df.merge(
            team_stats[['FullDate', 'Team', 'AvgPointsScored', 'AvgPointsAllowed']],
            left_on=['FullDate', 'HomeTeam'],
            right_on=['FullDate', 'Team'],
            how='left'
        ).rename(columns={
            'AvgPointsScored': 'HomeRollingScore',
            'AvgPointsAllowed': 'HomeRollingDefense'
        }).drop(columns=['Team']) # CHANGED: No need to drop 'Date', it merged on FullDate automatically

        # Merge Away Stats
        # CHANGED: right_on uses 'FullDate'
        self.df = self.df.merge(
            team_stats[['FullDate', 'Team', 'AvgPointsScored', 'AvgPointsAllowed']],
            left_on=['FullDate', 'AwayTeam'],
            right_on=['FullDate', 'Team'],
            how='left'
        ).rename(columns={
            'AvgPointsScored': 'AwayRollingScore',
            'AvgPointsAllowed': 'AwayRollingDefense'
        }).drop(columns=['Team']) # CHANGED: No need to drop 'Date'

        # Fill NaNs for the first few games
        self.df.fillna(0, inplace=True)
        print("Rolling stats created.")
        return self.df

    #Sharavv
    def create_h2h_stats(self):
        """
        Calculates the historical win rate of HomeTeam vs AwayTeam.
        """
        print("Calculating Head-to-Head Stats (Shaarav)...")

        # 1. Sort by Date to ensure we don't "cheat" by looking at future games
        self.df = self.df.sort_values('Date')

        # 2. History Dictionary
        # Key: A sorted tuple of teams, e.g., ("Cowboys", "Giants")
        # Value: {'TeamA_Wins': 0, 'TeamB_Wins': 0}
        history = {}

        # We need a list to store the results row-by-row
        h2h_win_rates = []

        # 3. Iterate through every game
        for index, row in self.df.iterrows():
            home = row['HomeTeam']
            away = row['AwayTeam']

            # Create a unique key for this matchup
            # Sorted so "Cowboys vs Giants" is the same key as "Giants vs Cowboys"
            matchup_key = tuple(sorted([home, away]))

            # --- A. PREDICT (Check history BEFORE this game) ---
            if matchup_key not in history:
                # No history yet? Assume 50% win rate
                history[matchup_key] = {home: 0, away: 0}
                win_pct = 0.5
            else:
                stats = history[matchup_key]

                # Make sure both teams exist in the stats (in case of new team names)
                if home not in stats: stats[home] = 0
                if away not in stats: stats[away] = 0

                total_games = stats[home] + stats[away]

                if total_games > 0:
                    win_pct = stats[home] / total_games
                else:
                    win_pct = 0.5

            # Save the Home Team's historical win % against this opponent
            h2h_win_rates.append(win_pct)

            # --- B. LEARN (Update history AFTER this game) ---
            # We use the result of this game to update the dictionary for the FUTURE
            if row['HomeWin'] == 1:
                history[matchup_key][home] += 1
            elif row['AwayWin'] == 1:
                history[matchup_key][away] += 1
            # (If it's a tie, we don't add a win to either side)

        # 4. Add the new column to the dataframe
        self.df['H2H_HomeWinPct'] = h2h_win_rates

        print("H2H stats calculated successfully.")
        return self.df

    #Brandon
    #1. MERGE BACK TO MAIN DATASET
    #2. Upload the upadated csv into github
    def save_file(self, new_filename):
        self.df.to_csv(new_filename, index=False)
        print(f"Success! Saved the full file to: {new_filename}")

    def run_logistic_regression(self):
        print("\n--- Running Logistic Regression ---")

        # 1. Select Features
        # Using the features we engineered: Rolling stats, Rest days, H2H, and Season records
        feature_cols = [
            'HomeRollingScore', 'HomeRollingDefense',
            'AwayRollingScore', 'AwayRollingDefense',
            'HomeRestDays', 'AwayRestDays',
            'H2H_HomeWinPct',
            'HomeWins', 'HomeLosses', # From DataCleaner
            'AwayWins', 'AwayLosses'
        ]

        # Filter dataframe to ensure we have no missing values in these columns
        model_df = self.df.dropna(subset=feature_cols)

        X = model_df[feature_cols]
        y = model_df['HomeWin']

        # 2. Split Data (80% Train, 20% Test)
        # shuffle=False is often better for time-series (train on past, test on future)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, shuffle=False)

        # 3. Scale Data (Important for Logistic Regression)
        scaler = StandardScaler()
        X_train_scaled = scaler.fit_transform(X_train)
        X_test_scaled = scaler.transform(X_test)

        # 4. Train Model
        model = LogisticRegression()
        model.fit(X_train_scaled, y_train)

        # 5. Evaluate
        preds = model.predict(X_test_scaled)
        acc = accuracy_score(y_test, preds)

        print(f"Model Accuracy: {acc:.4f}")
        print("\nConfusion Matrix:")
        print(confusion_matrix(y_test, preds))

        print("\nFeature Importance (Coefficients):")
        coeffs = pd.DataFrame({
            'Feature': feature_cols,
            'Coefficient': model.coef_[0]
        }).sort_values(by='Coefficient', ascending=False)
        print(coeffs)

        return model

FE = FeatureEngineer("2017-2025_scores_cleaned.csv")
FE.df.head()
FE.create_h2h_stats()
FE.create_rolling_stats()
FE.save_file("2017-2025_engineered.csv")
model = FE.run_logistic_regression()


Calculating Head-to-Head Stats (Shaarav)...
H2H stats calculated successfully.
Calculating 5-game rolling stats...
Rolling stats created.
Success! Saved the full file to: 2017-2025_engineered.csv

--- Running Logistic Regression ---
Model Accuracy: 0.7219

Confusion Matrix:
[[200  91]
 [ 50 166]]

Feature Importance (Coefficients):
               Feature  Coefficient
7             HomeWins     1.433891
10          AwayLosses     1.017817
2     AwayRollingScore     0.349548
1   HomeRollingDefense     0.276144
4         HomeRestDays     0.083136
6       H2H_HomeWinPct    -0.113527
5         AwayRestDays    -0.120631
0     HomeRollingScore    -0.144983
3   AwayRollingDefense    -0.248154
8           HomeLosses    -0.880198
9             AwayWins    -1.437834


/tmp/ipykernel_270/2215151253.py:69: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value '0' has dtype incompatible with datetime64[ns], please explicitly cast to a compatible dtype first.
  self.df.fillna(0, inplace=True)


**Feature Engineering Gb**

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score

class ModelEngineer:
    def __init__(self, features_csv):
        # 1. Load the dataset that contains all your new engineered features
        print(f"Loading data from {features_csv}...")
        self.df = pd.read_csv(features_csv)

    def train_gradient_boosting(self):
        print("\n--- Training Gradient Boosting Model ---")

        # 2. DEFINE TARGET (What we are predicting)
        # We want to predict if the Home Team won (1 = Yes, 0 = No)
        y = self.df['HomeWin']

        # 3. DEFINE FEATURES (What the model studies)
        # Drop columns that are text (names, dates) OR columns that "cheat" (Scores)
        # Adjust these column names if your team named them slightly differently!
        columns_to_drop = [
            'HomeWin', 'AwayWin',       # The answers
            'HomeScore', 'AwayScore',   # The points (cheating!)
            'Date', 'Season', 'Week',   # Dates
            'HomeTeam', 'AwayTeam',     # Names
            'GameStatus', 'Day',
            'FullDate'          # Unneeded text. Added FullDate to be dropped.
        ]

        # Keep only the columns that actually exist in your dataframe to avoid errors
        cols_to_drop_safe = [col for col in columns_to_drop if col in self.df.columns]
        X = self.df.drop(columns=cols_to_drop_safe)

        # Fill any missing values with 0 just in case
        X = X.fillna(0)

        print(f"Features the model is using: {list(X.columns)}")

        # 4. TRAIN / TEST SPLIT
        # We use 80% of data to train, and keep 20% to test accuracy
        # random_state=42 ensures you get the same random split every time you run it
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        print(f"Training on {len(X_train)} games. Testing on {len(X_test)} games.")

        # 5. INITIALIZE THE MODEL
        # This is where Gradient Boosting is created!
        gb_model = GradientBoostingClassifier(
            n_estimators=100,  # Number of trees to build
            learning_rate=0.1, # How much each tree contributes
            max_depth=3,       # How complex each tree can get
            random_state=42
        )

        # 6. TRAIN THE MODEL
        print("Training in progress... Please wait.")
        gb_model.fit(X_train, y_train)

        # 7. MAKE PREDICTIONS ON THE TEST SET
        predictions = gb_model.predict(X_test)

        # 8. PRINT ACCURACY PERCENTAGE (YOUR SPECIFIC TASK)
        accuracy = accuracy_score(y_test, predictions)
        accuracy_percentage = accuracy * 100

        print("\n" + "="*35)
        print(f" MODEL ACCURACY: {accuracy_percentage:.2f}%")
        print("="*35 + "\n")

        return gb_model

# ==========================================
# HOW TO RUN YOUR CODE
# ==========================================
model_engine = ModelEngineer('2017-2025_engineered.csv')
my_model = model_engine.train_gradient_boosting()


Loading data from 2017-2025_engineered.csv...

--- Training Gradient Boosting Model ---
Features the model is using: ['AwaySeeding', 'HomeSeeding', 'PostSeason', 'AwayWins', 'AwayLosses', 'HomeWins', 'HomeLosses', 'HomeRestDays', 'AwayRestDays', 'H2H_HomeWinPct', 'HomeRollingScore', 'HomeRollingDefense', 'AwayRollingScore', 'AwayRollingDefense']
Training on 2024 games. Testing on 507 games.
Training in progress... Please wait.

 MODEL ACCURACY: 81.07%



In [ ]:
import pandas as pd
from datetime import datetime, timedelta

class DataIntegrator:
    def __init__(self, history_file, raw_file):
        print(f"Loading history from {history_file}...")
        self.history_df = pd.read_csv(history_file)

        print(f"Loading raw schedule from {raw_file}...")
        self.raw_df = pd.read_csv(raw_file)

    def create_master_timeline(self):
        print("Creating Master Timeline...")

        # 1. PREPARE HISTORY
        # We use errors='coerce' to turn "0" into NaT (Not a Time)
        # We then drop rows where the date is broken.
        if 'FullDate' in self.history_df.columns:
            self.history_df['FullDate'] = pd.to_datetime(self.history_df['FullDate'], errors='coerce')
        elif 'Date' in self.history_df.columns:
             self.history_df['FullDate'] = pd.to_datetime(self.history_df['Date'], errors='coerce')

        # Remove the bad rows (the ones that were 0)
        original_count = len(self.history_df)
        self.history_df = self.history_df.dropna(subset=['FullDate'])
        dropped_count = original_count - len(self.history_df)
        if dropped_count > 0:
            print(f"Warning: Dropped {dropped_count} rows with invalid dates (likely '0').")

        history_subset = self.history_df.copy()

        # 2. PREPARE UPCOMING SCHEDULE
        upcoming_games = self.raw_df[self.raw_df['GameStatus'] == 'UPCOMING'].copy()

        def fix_upcoming_date(row):
            try:
                # Combine Date (e.g., "11/20") and Season (e.g., 2025)
                date_str = f"{row['Date']}/{row['Season']}"
                return pd.to_datetime(date_str, format='%m/%d/%Y')
            except:
                # Fallback: If date is TBD, set it to 7 days from now
                return datetime.now() + timedelta(days=7)

        upcoming_games['FullDate'] = upcoming_games.apply(fix_upcoming_date, axis=1)

        # Fill missing feature columns in upcoming games with 0
        for col in history_subset.columns:
            if col not in upcoming_games.columns:
                upcoming_games[col] = 0

        # 3. MERGE
        master_df = pd.concat([history_subset, upcoming_games], ignore_index=True)

        # Sort by Date (Oldest -> Newest)
        master_df = master_df.sort_values('FullDate')

        # Forward Fill stats (Optional: helps fill 0s in upcoming games with recent real stats)
        # master_df = master_df.fillna(method='ffill')

        # 4. SAVE
        output_filename = 'combined_schedule.csv'
        master_df.to_csv(output_filename, index=False)
        print(f"Master timeline created and saved to {output_filename}")

        return master_df

# ==========================================
# RUN CELL 4
# ==========================================
integrator = DataIntegrator("2017-2025_engineered.csv", "2017-2025_scores.csv")
master_timeline = integrator.create_master_timeline()

# Check the tail to ensure upcoming games are there
print(master_timeline[['FullDate', 'HomeTeam', 'AwayTeam', 'GameStatus']].tail())

Loading history from 2017-2025_engineered.csv...
Loading raw schedule from 2017-2025_scores.csv...
Creating Master Timeline...
Master timeline created and saved to combined_schedule.csv
                       FullDate    HomeTeam    AwayTeam GameStatus
2561 2026-03-05 03:39:40.755505      Giants     Cowboys   UPCOMING
2562 2026-03-05 03:39:40.755578      Eagles  Commanders   UPCOMING
2563 2026-03-05 03:39:40.755651    Steelers      Ravens   UPCOMING
2564 2026-03-05 03:39:40.755721       49ers    Seahawks   UPCOMING
2565 2026-03-05 03:39:40.755791  Buccaneers    Panthers   UPCOMING


In [ ]:
import pandas as pd
import numpy as np

class PredictionFeatureEngineer:
    def __init__(self, combined_file):
        # TODO: Load 'combined_schedule.csv'
        print(f"Loading schedule from {combined_file}...")
        self.df = pd.read_csv(combined_file)

        # Ensure 'FullDate' is a datetime object and sort chronologically
        # This is critical so our rolling stats don't accidentally look into the future
        self.df['FullDate'] = pd.to_datetime(self.df['FullDate'])
        self.df = self.df.sort_values('FullDate').reset_index(drop=True)

    def add_features(self):
        print("Adding prediction features for upcoming games...")

        # 1. RESHAPE TO LONG FORMAT
        # Stack Home and Away games so you have a list of (Date, Team, PointsScored)
        home_df = self.df[['FullDate', 'HomeTeam', 'HomeScore', 'AwayScore']].copy()
        home_df.columns = ['FullDate', 'Team', 'PointsScored', 'PointsAllowed']

        away_df = self.df[['FullDate', 'AwayTeam', 'AwayScore', 'HomeScore']].copy()
        away_df.columns = ['FullDate', 'Team', 'PointsScored', 'PointsAllowed']

        # Combine both datasets and re-sort by date
        long_df = pd.concat([home_df, away_df]).sort_values('FullDate').reset_index(drop=True)

        # 2. CALCULATE ROLLING STATS
        # Group by Team, Calculate Rolling Mean (Window=5)
        def calc_rolling(group):
            # CRITICAL: Use .shift(1) so you don't include the current game in the average
            # min_periods=1 ensures early season games still get an average
            group['AvgPointsScored'] = group['PointsScored'].shift(1).rolling(window=5, min_periods=1).mean()
            group['AvgPointsAllowed'] = group['PointsAllowed'].shift(1).rolling(window=5, min_periods=1).mean()
            return group

        long_df = long_df.groupby('Team', group_keys=False).apply(calc_rolling)

        # Extract just the columns we need to merge back
        stats_df = long_df[['FullDate', 'Team', 'AvgPointsScored', 'AvgPointsAllowed']]

        # 3. MERGE STATS BACK
        # Merge the new 'AvgPoints' columns back into the main schedule for the Home Team
        self.df = pd.merge(self.df, stats_df,
                           left_on=['FullDate', 'HomeTeam'],
                           right_on=['FullDate', 'Team'],
                           how='left').drop(columns=['Team'])
        self.df.rename(columns={'AvgPointsScored': 'Home_AvgPointsScored',
                                'AvgPointsAllowed': 'Home_AvgPointsAllowed'}, inplace=True)

        # Map them to the Away Team
        self.df = pd.merge(self.df, stats_df,
                           left_on=['FullDate', 'AwayTeam'],
                           right_on=['FullDate', 'Team'],
                           how='left').drop(columns=['Team'])
        self.df.rename(columns={'AvgPointsScored': 'Away_AvgPointsScored',
                                'AvgPointsAllowed': 'Away_AvgPointsAllowed'}, inplace=True)

        # 4. CALCULATE MATCHUP HISTORY (H2H)
        # Create a unique ID for each matchup (e.g., "Bears-Packers")
        # Sorting guarantees "Cowboys-Giants" is the exact same ID as "Giants-Cowboys"
        self.df['MatchupID'] = self.df.apply(
            lambda row: "-".join(sorted([str(row['HomeTeam']), str(row['AwayTeam'])])), axis=1
        )

        # Figure out who won this game
        self.df['Winner'] = np.where(self.df['HomeScore'] > self.df['AwayScore'], self.df['HomeTeam'],
                            np.where(self.df['AwayScore'] > self.df['HomeScore'], self.df['AwayTeam'], 'Tie'))

        # Check if Home Team won the *previous* time this ID appeared
        self.df['LastWinner'] = self.df.groupby('MatchupID')['Winner'].shift(1)
        self.df['HomeWonLastH2H'] = (self.df['HomeTeam'] == self.df['LastWinner']).astype(int)

        # Drop the temporary calculation columns to keep the dataframe clean
        self.df.drop(columns=['Winner', 'LastWinner', 'MatchupID'], inplace=True)

        # 5. SAVE
        # Save to 'prediction_ready.csv'
        output_file = 'prediction_ready.csv'
        self.df.to_csv(output_file, index=False)
        print(f"Features added to upcoming games! Saved successfully to '{output_file}'.")

        return self.df

# ==========================================
# HOW TO RUN YOUR CODE
# ==========================================
pfe = PredictionFeatureEngineer('combined_schedule.csv')
prediction_ready_df = pfe.add_features()

# Optional: View the last 5 upcoming games to ensure the stats properly filled
print("\nSnapshot of upcoming games (Features Generated):")
print(prediction_ready_df[prediction_ready_df['GameStatus'] == 'UPCOMING'] \
    [['FullDate', 'HomeTeam', 'AwayTeam', 'Home_AvgPointsScored']].head())

Loading schedule from combined_schedule.csv...
Adding prediction features for upcoming games...
Features added to upcoming games! Saved successfully to 'prediction_ready.csv'.

Snapshot of upcoming games (Features Generated):
       FullDate HomeTeam AwayTeam  Home_AvgPointsScored
2373 2025-11-20   Texans    Bills                  22.4
2374 2025-11-20   Texans    Bills                  22.4
2375 2025-11-20   Texans    Bills                  18.6
2376 2025-11-20   Texans    Bills                  18.6
2377 2025-11-20   Texans    Bills                  22.4


/tmp/ipython-input-547/2101939757.py:38: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  long_df = long_df.groupby('Team', group_keys=False).apply(calc_rolling)


In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report

class GamePredictor:
    def __init__(self, data_file):
        print(f"Loading data from '{data_file}'...")
        self.df = pd.read_csv(data_file)
        self.model = None

        # Define the base features we know we generated
        self.features = [
            'Home_AvgPointsScored', 'Home_AvgPointsAllowed',
            'Away_AvgPointsScored', 'Away_AvgPointsAllowed',
            'HomeWonLastH2H'
        ]

        # Safely add older features (like Rest Days or Records) if they survived the pipeline
        optional_features = ['HomeRestDays', 'AwayRestDays', 'HomeWins', 'HomeLosses', 'AwayWins', 'AwayLosses']
        for f in optional_features:
            if f in self.df.columns:
                self.features.append(f)

    def train_model(self):
        print("\n--- Training Random Forest Model ---")

        # 1. Isolate historical games (where we actually have a winner)
        # We drop any rows where 'HomeWin' is missing, just to be safe
        historical_df = self.df[self.df['GameStatus'] != 'UPCOMING'].dropna(subset=['HomeWin'] + self.features)

        X = historical_df[self.features]
        y = historical_df['HomeWin']

        # 2. Train/Test Split (80/20)
        X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

        # 3. Initialize and Train the Random Forest
        self.model = RandomForestClassifier(
            n_estimators=100,      # Number of trees in the forest
            max_depth=5,           # Limit depth to prevent overfitting
            min_samples_split=4,   # Minimum samples required to split an internal node
            random_state=42
        )
        self.model.fit(X_train, y_train)

        # 4. Evaluate Accuracy
        predictions = self.model.predict(X_test)
        accuracy = accuracy_score(y_test, predictions)

        print(f"Model Accuracy on Test Data: {accuracy * 100:.2f}%\n")

        # Show which features the Random Forest found most important
        importances = pd.DataFrame({
            'Feature': self.features,
            'Importance': self.model.feature_importances_
        }).sort_values('Importance', ascending=False)
        print("Feature Importances:")
        print(importances.to_string(index=False))

    def predict_upcoming_games(self):
        print("\n--- Predicting Upcoming Games ---")

        if self.model is None:
            print("Error: You must run train_model() before predicting!")
            return

        # 1. Isolate and DEDUPLICATE the upcoming schedule
        # We group by Date and Teams to ensure each game only appears once
        upcoming_df = self.df[self.df['GameStatus'] == 'UPCOMING'].copy()

        # This line removes the duplicates, keeping the most recent/complete entry
        upcoming_df = upcoming_df.drop_duplicates(subset=['FullDate', 'AwayTeam', 'HomeTeam'], keep='last')

        if upcoming_df.empty:
            print("No upcoming games found in the dataset.")
            return

        # ... rest of your code remains the same ...
        X_upcoming = upcoming_df[self.features].fillna(0)
        upcoming_df['Predicted_HomeWin'] = self.model.predict(X_upcoming)
        probabilities = self.model.predict_proba(X_upcoming)

        upcoming_df['Confidence'] = probabilities.max(axis=1) * 100
        upcoming_df['Projected_Winner'] = upcoming_df.apply(
            lambda row: row['HomeTeam'] if row['Predicted_HomeWin'] == 1 else row['AwayTeam'],
            axis=1
        )

        results = upcoming_df[['FullDate', 'AwayTeam', 'HomeTeam', 'Projected_Winner', 'Confidence']]
        print("\n Official Predictions:")
        print(results.to_string(index=False))
        return results


predictor = GamePredictor('prediction_ready.csv')
predictor.train_model()
final_picks = predictor.predict_upcoming_games()

Loading data from 'prediction_ready.csv'...

--- Training Random Forest Model ---
Model Accuracy on Test Data: 78.94%

Feature Importances:
              Feature  Importance
             HomeWins    0.255660
           AwayLosses    0.237411
           HomeLosses    0.178734
             AwayWins    0.172735
 Away_AvgPointsScored    0.050552
 Home_AvgPointsScored    0.044513
Home_AvgPointsAllowed    0.017360
         AwayRestDays    0.015765
         HomeRestDays    0.014252
Away_AvgPointsAllowed    0.012249
       HomeWonLastH2H    0.000769

--- Predicting Upcoming Games ---

 Official Predictions:
                  FullDate   AwayTeam   HomeTeam Projected_Winner  Confidence
                2025-11-20      Bills     Texans            Bills   61.420749
                2025-11-23 Buccaneers       Rams             Rams   75.900346
                2025-11-23   Steelers      Bears            Bears   61.795971
                2025-11-23   Patriots    Bengals         Patriots   70.220228
   